# 多模态检索工程：从双编码器、对比学习到可授权召回

多模态检索不是把图片和文本各自变成向量后直接调用向量库。本 Notebook 用可控数值特征从零实现一条 image-text 双编码链路：配对数据与切分、训练集统计、对称 InfoNCE、向量归一化、双向检索、Recall/MRR、hard negative、ACL、索引版本和批量 API。

## 学习目标

1. 写清 image/text 输入、pair、family、tenant 与版本合同；
2. 从零推导并实现对称对比损失及线性双编码器梯度；
3. 区分 pair-level 命中、语义相关和 false negative；
4. 实现 image→text、text→image 的 Recall@K 与 MRR；
5. 把授权过滤、零向量、索引快照、trace 和回滚纳入检索系统。

> 数据是线性生成的受控教学样本，容易被线性模型学习。这里的高分只验证代码与工程合同，不能代表真实图片、长文本、OCR 或跨语言上的 VLM 能力。

## 1. 先定义 pair 与服务合同

训练记录至少包含 `pair_id、image_id、caption_id、family_id、tenant、allowed_roles、image_version、text_version、label_source`。一张图可能有多条正确描述，一条描述也可能适配多张近似图片，因此 batch 对角线只能在“一对一受控数据”中直接当唯一正例。

在线接口不直接相信请求里的 tenant；认证层生成身份上下文。索引返回稳定 item ID、相似度、模型/预处理/索引版本和可回放 trace。删除图片时还要删除派生 embedding、缓存和训练样本引用。

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
import hashlib
import json
import math
import numpy as np

RNG = np.random.default_rng(17)
CATEGORIES = ("路由器", "相机", "鞋", "咖啡", "汽车", "植物")
N, LATENT_DIM, IMAGE_DIM, TEXT_DIM = 48, 6, 10, 11
category_id = np.arange(N) % len(CATEGORIES)
position_in_category = np.arange(N) // len(CATEGORIES)
latent = 1.2 * np.eye(LATENT_DIM)[category_id] + RNG.normal(0, 0.45, (N, LATENT_DIM))
image_mixer = RNG.normal(0, 1, (LATENT_DIM, IMAGE_DIM))
text_mixer = RNG.normal(0, 1, (LATENT_DIM, TEXT_DIM))
image_raw = latent @ image_mixer + RNG.normal(0, 0.08, (N, IMAGE_DIM))
text_raw = latent @ text_mixer + RNG.normal(0, 0.08, (N, TEXT_DIM))

@dataclass(frozen=True)
class PairRecord:
    pair_id: str
    image_id: str
    caption_id: str
    family_id: str
    category: str
    tenant: str
    allowed_roles: frozenset[str]
    caption: str

records = [
    PairRecord(
        f"pair-{i:03d}", f"img-{i:03d}", f"cap-{i:03d}", f"family-{i:03d}",
        CATEGORIES[category_id[i]], "tenant-a" if i < 42 else "tenant-secret",
        frozenset({"employee"}) if i < 42 else frozenset({"admin"}),
        f"{CATEGORIES[category_id[i]]} 的受控描述，样本 {i:03d}",
    ) for i in range(N)
]
assert len({r.pair_id for r in records}) == N
print("pairs=", N, "image_dim=", IMAGE_DIM, "text_dim=", TEXT_DIM)


## 2. 切分与预处理：family 先隔离，统计量只看 train

同一原图的裁剪、压缩版和多 caption 必须按 family 进入同一 split，否则模型只是在识别近重复。这里每类前 5 个样本用于训练，第 6 个用于 validation，最后 2 个用于 test；三者 family 完全不交叉。

真实图像还要锁定颜色空间、EXIF 旋转、alpha、resize/crop、像素范围和图像解码库版本；文本要锁定 Unicode、模板、语言检测和 tokenizer。任何均值、方差或 PCA 都只能在 train 上拟合。

In [ ]:
train_idx = np.flatnonzero(position_in_category < 5)
valid_idx = np.flatnonzero(position_in_category == 5)
test_idx = np.flatnonzero(position_in_category >= 6)

def fit_standardizer(matrix: np.ndarray, indices: np.ndarray):
    mean = matrix[indices].mean(axis=0)
    scale = matrix[indices].std(axis=0)
    scale = np.where(scale < 1e-8, 1.0, scale)
    return mean, scale

image_mean, image_scale = fit_standardizer(image_raw, train_idx)
text_mean, text_scale = fit_standardizer(text_raw, train_idx)
image_x = (image_raw - image_mean) / image_scale
text_x = (text_raw - text_mean) / text_scale

family_sets = [{records[i].family_id for i in ids} for ids in (train_idx, valid_idx, test_idx)]
assert family_sets[0].isdisjoint(family_sets[1] | family_sets[2])
assert family_sets[1].isdisjoint(family_sets[2])
assert len(train_idx) == 30 and len(valid_idx) == 6 and len(test_idx) == 12
print({"train": len(train_idx), "validation": len(valid_idx), "test": len(test_idx)})


## 3. 双编码器与对称 InfoNCE

图像编码器与文本编码器分别输出单位向量 $z_i,z_t$。batch 相似度为 $s_{ab}=z^i_a\cdot z^t_b/\tau$，$\tau$ 是温度。对称损失同时做 image→text 和 text→image 的交叉熵：

$$L=\frac12\left[-\frac1B\sum_a\log p(t_a|i_a)-\frac1B\sum_a\log p(i_a|t_a)\right].$$

下面手写 softmax、归一化反向传播和两个线性投影的梯度。生产模型通常是 CNN/ViT 与 Transformer，且需要大规模分布式负例、mixed precision、梯度同步和数据治理。

In [ ]:
def row_normalize(values: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    return values / np.maximum(norms, 1e-12)

def softmax_rows(logits: np.ndarray) -> np.ndarray:
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)

def contrastive_loss_and_grad(xi, xt, wi, wt, temperature=0.12, l2=1e-4):
    ui, vt = xi @ wi, xt @ wt
    zi, zt = row_normalize(ui), row_normalize(vt)
    logits = zi @ zt.T / temperature
    batch = len(xi)
    identity = np.eye(batch)
    prob_i, prob_t = softmax_rows(logits), softmax_rows(logits.T)
    data_loss = (-np.log(np.diag(prob_i) + 1e-12).mean()
                 -np.log(np.diag(prob_t) + 1e-12).mean()) / 2
    loss = data_loss + 0.5 * l2 * (np.sum(wi * wi) + np.sum(wt * wt))
    grad_logits = ((prob_i - identity) + (prob_t - identity).T) / (2 * batch * temperature)
    grad_zi, grad_zt = grad_logits @ zt, grad_logits.T @ zi
    ui_norm = np.maximum(np.linalg.norm(ui, axis=1, keepdims=True), 1e-12)
    vt_norm = np.maximum(np.linalg.norm(vt, axis=1, keepdims=True), 1e-12)
    grad_ui = (grad_zi - zi * np.sum(grad_zi * zi, axis=1, keepdims=True)) / ui_norm
    grad_vt = (grad_zt - zt * np.sum(grad_zt * zt, axis=1, keepdims=True)) / vt_norm
    return loss, xi.T @ grad_ui + l2 * wi, xt.T @ grad_vt + l2 * wt

probe = np.eye(3)
assert np.allclose(softmax_rows(probe).sum(axis=1), 1.0)
assert np.allclose(np.linalg.norm(row_normalize(probe), axis=1), 1.0)


## 4. 训练：只优化 train，validation 留给模型选择

本例为了聚焦梯度实现，预先固定 embedding 维度、温度、学习率计划和 epoch，因此 validation 只被保留而不参与调参；这比看过 test 后改参数更诚实，但也不能据此宣称超参数已优化。真实项目只能用 validation 选择 checkpoint/温度/维度，并记录数据快照、随机种子、batch 组成、optimizer 与曲线，test 冻结后只报告一次。batch 中若有同义 caption 或同类图片，错误地把它们都当负例会制造 false negative。

In [ ]:
EMBED_DIM = 8
wi = RNG.normal(0, 0.15, (IMAGE_DIM, EMBED_DIM))
wt = RNG.normal(0, 0.15, (TEXT_DIM, EMBED_DIM))
loss_history = []
for epoch in range(1000):
    loss, grad_i, grad_t = contrastive_loss_and_grad(
        image_x[train_idx], text_x[train_idx], wi, wt
    )
    learning_rate = 0.12 * (0.999 ** epoch)
    wi -= learning_rate * grad_i
    wt -= learning_rate * grad_t
    loss_history.append(float(loss))

assert loss_history[-1] < loss_history[0] * 0.1
print("loss:", round(loss_history[0], 4), "->", round(loss_history[-1], 4))


## 5. 向量索引与稳定排序

单位向量的内积等于余弦相似度。教学规模用全量矩阵乘法；生产可替换为 Faiss/HNSW/向量数据库，但必须评估 ANN Recall、延迟、内存和删除一致性。相同分数用稳定 item ID 打破并列，不能依赖分片返回顺序。零向量必须拒绝，而不是被归一化成 NaN。

In [ ]:
image_embedding = row_normalize(image_x @ wi)
text_embedding = row_normalize(text_x @ wt)

def stable_rank(query: np.ndarray, matrix: np.ndarray, ids: list[str], top_k: int):
    if not 1 <= top_k <= len(ids):
        raise ValueError("top_k 越界")
    if np.linalg.norm(query) < 1e-12:
        return []
    query = query / np.linalg.norm(query)
    scores = matrix @ query
    order = sorted(range(len(ids)), key=lambda j: (-float(scores[j]), ids[j]))[:top_k]
    return [(ids[j], float(scores[j])) for j in order]

all_image_ids = [record.image_id for record in records]
all_caption_ids = [record.caption_id for record in records]
example = stable_rank(text_embedding[test_idx[0]], image_embedding, all_image_ids, 5)
print(example)
assert example


## 6. 双向评估：Recall@K 与 MRR

对每个 test query，只在同一 test gallery 中评价，避免 train 近重复让结果虚高。Recall@K 表示正确配对是否进入前 K；MRR 强调第一个正确结果的位置。多正例任务要把 qrels 改成集合，不能只认对角线；还应按语言、图像质量、长尾类别、OCR 文本和敏感属性分桶。

In [ ]:
def paired_ranks(query_matrix: np.ndarray, gallery_matrix: np.ndarray) -> list[int]:
    similarity = query_matrix @ gallery_matrix.T
    ranks = []
    for row in range(len(query_matrix)):
        order = np.argsort(-similarity[row], kind="stable")
        ranks.append(int(np.flatnonzero(order == row)[0]) + 1)
    return ranks

def retrieval_metrics(ranks: list[int]) -> dict[str, float]:
    values = np.asarray(ranks)
    return {
        "Recall@1": float(np.mean(values <= 1)),
        "Recall@3": float(np.mean(values <= 3)),
        "MRR": float(np.mean(1 / values)),
    }

i2t_ranks = paired_ranks(image_embedding[test_idx], text_embedding[test_idx])
t2i_ranks = paired_ranks(text_embedding[test_idx], image_embedding[test_idx])
metrics = {"image_to_text": retrieval_metrics(i2t_ranks), "text_to_image": retrieval_metrics(t2i_ranks)}
print(json.dumps(metrics, ensure_ascii=False, indent=2))
assert metrics["image_to_text"]["Recall@3"] >= 0.9
assert metrics["text_to_image"]["Recall@3"] >= 0.9


## 7. Hard negative 与 false negative

随机负例往往过于容易；同类但属性不同的样本更能训练细粒度区分。然而“同类”也可能是合法多正例。挖掘前必须用 family、重复检测、人工标签或多 caption qrels 排除潜在正例。线上点击未发生也不等于负例，因为用户可能没看到该结果。

In [ ]:
def mine_hard_negatives(indices: np.ndarray, top_n=2):
    similarity = image_embedding[indices] @ text_embedding[indices].T
    result = {}
    for local, global_index in enumerate(indices):
        candidates = [
            other for other in range(len(indices))
            if other != local and category_id[indices[other]] == category_id[global_index]
        ]
        candidates.sort(key=lambda other: (-float(similarity[local, other]), records[indices[other]].caption_id))
        result[records[global_index].image_id] = [records[indices[o]].caption_id for o in candidates[:top_n]]
    return result

hard_negatives = mine_hard_negatives(train_idx)
assert all(len(values) <= 2 for values in hard_negatives.values())
assert all(image_id.replace("img", "cap") not in negatives for image_id, negatives in hard_negatives.items())
print(list(hard_negatives.items())[:3])


## 8. ACL 必须在候选空间里生效

若先全库 ANN top-k 再删越权结果，秘密图片会占候选预算，也可能通过分数、耗时或缓存泄漏。高隔离租户应使用物理分区索引；共享索引至少要支持原生 pre-filter。缓存键包含身份权限版本、query embedding/model、filters 与 index generation。多模态内容还需处理人脸、地理位置、版权和有害内容策略。

In [ ]:
_ISSUER = object()

@dataclass(frozen=True)
class AuthContext:
    tenant: str
    roles: frozenset[str]
    subject: str
    _marker: object = field(repr=False, compare=False)

def authenticate_demo(token: str) -> AuthContext:
    if token != "signed-tenant-a":
        raise PermissionError("认证失败")
    return AuthContext("tenant-a", frozenset({"employee"}), "user-7", _ISSUER)

def authorized_image_search(query_embedding: np.ndarray, auth: AuthContext, top_k=5):
    if not isinstance(auth, AuthContext) or auth._marker is not _ISSUER:
        raise PermissionError("身份上下文不可信")
    if top_k < 1:
        raise ValueError("top_k 必须为正整数")
    visible = [
        i for i, record in enumerate(records)
        if record.tenant == auth.tenant and record.allowed_roles & auth.roles
    ]
    if not visible:
        return [], {"visible_count": 0, "degraded": False, "reason": "no_authorized_items"}
    ids = [records[i].image_id for i in visible]
    ranked = stable_rank(query_embedding, image_embedding[visible], ids, min(top_k, len(ids)))
    return ranked, {"visible_count": len(visible), "degraded": False}

AUTH = authenticate_demo("signed-tenant-a")
authorized, safe_trace = authorized_image_search(text_embedding[0], AUTH)
assert all(int(item_id.split("-")[1]) < 42 for item_id, _ in authorized)
print(authorized[:3], safe_trace)


## 9. 发布、观测与降级

模型包至少绑定 image preprocess、text tokenizer/template、两个 encoder、归一化、embedding dimension、distance、训练数据 snapshot 和安全策略。升级任一编码器都要重建对应索引并做双写/影子回放。

线上监控 query/image 解码失败、零向量、各分桶 Recall、ANN recall、P50/P99、候选过滤率、无结果率、索引新鲜度和撤权传播。向量服务超时时可退回经过 ACL 的关键词/metadata 搜索，但必须标记 degraded，不能绕过授权。

In [ ]:
def array_hash(value: np.ndarray) -> str:
    return hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()

MODEL_MANIFEST = {
    "model_version": "linear-dual-encoder-v1",
    "image_preprocess": "standardize-train-snapshot-v1",
    "text_preprocess": "standardize-train-snapshot-v1",
    "embedding_dim": EMBED_DIM,
    "distance": "cosine-via-normalized-inner-product",
    "image_weight_sha256": array_hash(wi),
    "text_weight_sha256": array_hash(wt),
    "index_generation": "tenant-a-g1",
}
manifest_blob = json.dumps(MODEL_MANIFEST, sort_keys=True, ensure_ascii=False)
MODEL_MANIFEST["bundle_sha256"] = hashlib.sha256(manifest_blob.encode()).hexdigest()
assert len(MODEL_MANIFEST["bundle_sha256"]) == 64
print(MODEL_MANIFEST)


## 10. 最小工程回归

回归不只看 loss：还要锁定 split 隔离、train-only 统计、向量有限且单位化、双向 Recall、稳定 tie-break、零向量拒绝、ACL、伪造身份、版本指纹和删除语义。真实系统应增加损坏图片、极端尺寸、透明图、OCR 密集图、重复 caption、多语言和 adversarial prompt 测试。

In [ ]:
gradient_rng = np.random.default_rng(101)
check_xi = gradient_rng.normal(size=(4, 3))
check_xt = gradient_rng.normal(size=(4, 5))
check_wi = gradient_rng.normal(size=(3, 2))
check_wt = gradient_rng.normal(size=(5, 2))
_, analytic_wi, _ = contrastive_loss_and_grad(check_xi, check_xt, check_wi, check_wt)
epsilon = 1e-6
plus, minus = check_wi.copy(), check_wi.copy()
plus[0, 0] += epsilon
minus[0, 0] -= epsilon
loss_plus = contrastive_loss_and_grad(check_xi, check_xt, plus, check_wt)[0]
loss_minus = contrastive_loss_and_grad(check_xi, check_xt, minus, check_wt)[0]
numeric_gradient = (loss_plus - loss_minus) / (2 * epsilon)
assert math.isclose(numeric_gradient, analytic_wi[0, 0], rel_tol=1e-5, abs_tol=1e-5)
assert np.isfinite(image_embedding).all() and np.isfinite(text_embedding).all()
assert np.allclose(np.linalg.norm(image_embedding, axis=1), 1.0)
assert np.allclose(np.linalg.norm(text_embedding, axis=1), 1.0)
assert len(i2t_ranks) == len(test_idx) == len(t2i_ranks)
assert stable_rank(np.zeros(EMBED_DIM), image_embedding, all_image_ids, 3) == []
tie_query = np.array([1.0, 0.0])
tie_gallery = np.array([[1.0, 0.0], [1.0, 0.0], [0.0, 1.0]])
tie_result = stable_rank(tie_query, tie_gallery, ["item-b", "item-a", "item-c"], 3)
assert [item_id for item_id, _ in tie_result] == ["item-a", "item-b", "item-c"]
assert all(score <= 1.0 + 1e-9 for _, score in authorized)
assert safe_trace["visible_count"] == 42
no_access = AuthContext("tenant-a", frozenset({"visitor"}), "user-8", _ISSUER)
empty_result, empty_trace = authorized_image_search(text_embedding[0], no_access)
assert empty_result == [] and empty_trace["reason"] == "no_authorized_items"
try:
    authorized_image_search(text_embedding[0], AuthContext("tenant-a", frozenset({"employee"}), "x", object()))
    raise AssertionError("伪造身份必须失败")
except PermissionError:
    pass
try:
    stable_rank(text_embedding[0], image_embedding, all_image_ids, N + 1)
    raise AssertionError("非法 top_k 必须失败")
except ValueError:
    pass
assert MODEL_MANIFEST["index_generation"].startswith("tenant-a")
print("多模态切分、训练、检索、指标、ACL 与版本断言全部通过。")


## 11. 研究依据与生产边界

- Radford et al., [Learning Transferable Visual Models From Natural Language Supervision](https://arxiv.org/abs/2103.00020), 2021：CLIP 双编码器与自然语言监督。
- van den Oord et al., [Representation Learning with Contrastive Predictive Coding](https://arxiv.org/abs/1807.03748)：InfoNCE 的代表性来源。
- Douze et al., [The Faiss library](https://arxiv.org/abs/2401.08281)：向量索引方法与工程取舍。

教学实现只有线性数值 encoder、全 batch 负例和精确矩阵搜索；没有真实图像解码、tokenizer、ViT/Transformer、分布式训练、ANN 量化、内容审核或版权治理。生产替换模型时，仍应保留这里的 split、qrels、ACL、版本与冷启动回归合同。